# Level 3 — Data Quality Checks

## Objective

The objective of Level 3 is to clean and validate the processed train schedule dataset before performing further analysis and visualization.

### Tasks

1. Handle missing schedule values.
2. Remove duplicate train records.
3. Verify correct station order for each train route.
4. Save the verified dataset for subsequent analysis.

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("Level_2_Processed_Train_Schedule.csv")

df.head()

,SN,Train_No,Station_Code,1A,2A,3A,SL,Station_Name,Route_Number,Arrival_time,Departure_Time,Distance,Journey_Duration,Journey_Duration_Hours,Route_Type
0,1,107,SWV,100,100,100,100,SAWANTWADI R,1,1900-01-01 00:00:00,1900-01-01 10:25:00,0,0 days 01:45:00,1.750000,Medium
1,2,107,THVM,260,228,196,164,THIVIM,1,1900-01-01 11:06:00,1900-01-01 11:08:00,32,0 days 01:45:00,1.750000,Medium
2,3,107,KRMI,345,296,247,198,KARMALI,1,1900-01-01 11:28:00,1900-01-01 11:30:00,49,0 days 01:45:00,1.750000,Medium
3,4,107,MAO,490,412,334,256,MADGOAN JN.,1,1900-01-01 12:10:00,1900-01-01 00:00:00,78,0 days 01:45:00,1.750000,Medium
4,1,108,MAO,100,100,100,100,MADGOAN JN.,1,1900-01-01 00:00:00,1900-01-01 20:30:00,0,0 days 01:55:00,1.916667,Medium


In [3]:
df.head()

,SN,Train_No,Station_Code,1A,2A,3A,SL,Station_Name,Route_Number,Arrival_time,Departure_Time,Distance,Journey_Duration,Journey_Duration_Hours,Route_Type
0,1,107,SWV,100,100,100,100,SAWANTWADI R,1,1900-01-01 00:00:00,1900-01-01 10:25:00,0,0 days 01:45:00,1.750000,Medium
1,2,107,THVM,260,228,196,164,THIVIM,1,1900-01-01 11:06:00,1900-01-01 11:08:00,32,0 days 01:45:00,1.750000,Medium
2,3,107,KRMI,345,296,247,198,KARMALI,1,1900-01-01 11:28:00,1900-01-01 11:30:00,49,0 days 01:45:00,1.750000,Medium
3,4,107,MAO,490,412,334,256,MADGOAN JN.,1,1900-01-01 12:10:00,1900-01-01 00:00:00,78,0 days 01:45:00,1.750000,Medium
4,1,108,MAO,100,100,100,100,MADGOAN JN.,1,1900-01-01 00:00:00,1900-01-01 20:30:00,0,0 days 01:55:00,1.916667,Medium


In [4]:
df.shape

(186074, 15)

In [5]:
missing_values = df.isnull().sum()

missing_values

SN                        0
Train_No                  0
Station_Code              0
1A                        0
2A                        0
3A                        0
SL                        0
Station_Name              0
Route_Number              0
Arrival_time              0
Departure_Time            0
Distance                  0
Journey_Duration          0
Journey_Duration_Hours    0
Route_Type                0
dtype: int64

In [6]:
missing_values[missing_values > 0]

Series([], dtype: int64)

In [7]:
print("Total missing values:", df.isnull().sum().sum())

if df.isnull().sum().sum() == 0:
    print("No missing/null values found in the dataset.")

Total missing values: 0
No missing/null values found in the dataset.


##  Missing Schedule Values

- No null/NaN values were found in the processed dataset.
- Therefore, no rows were removed or values imputed for null values.
- The `00:00:00` schedule values were not treated as null values because they represent boundary schedule entries in certain train routes.
- These boundary time values were already handled during Level 2 journey-duration processing.

In [8]:
duplicate_count = df.duplicated().sum()

print("Total duplicate records:", duplicate_count)

Total duplicate records: 0


In [9]:
df[df.duplicated(keep=False)].head(20)

,SN,Train_No,Station_Code,1A,2A,3A,SL,Station_Name,Route_Number,Arrival_time,Departure_Time,Distance,Journey_Duration,Journey_Duration_Hours,Route_Type


## Duplicate Train Records

- Complete duplicate records were checked using all columns.
- The dataset contains **0 duplicate records**.
- Therefore, no records were removed during duplicate handling.
- Repeated `Train_No` values were retained because each train can have multiple station records, which represent its route sequence rather than duplicate records.

## Verify Station Order

The station sequence for each train is validated using the `SN` (sequence number) column.

The validation checks whether station sequence numbers are arranged correctly within each train route.

In [11]:
sequence_check = df.groupby("Train_No")["SN"].apply(
    lambda x: x.is_monotonic_increasing
)

sequence_check.value_counts()

SN
True    11113
Name: count, dtype: int64

## Station Order Validation

- The `SN` (station sequence number) column was used to verify the order of stations within each train route.
- All **11,113 trains** have station sequence numbers in increasing order.
- No incorrectly ordered train routes were identified.
- Therefore, no station records required reordering during this validation step.

In [12]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Missing values:", df.isnull().sum().sum())
print("Duplicate records:", df.duplicated().sum())
print("Trains:", df["Train_No"].nunique())

Rows: 186074
Columns: 15
Missing values: 0
Duplicate records: 0
Trains: 11113


In [13]:
df.to_csv(
    "Level_3_Verified_Train_Schedule.csv",
    index=False
)

print("Verified dataset saved successfully.")

Verified dataset saved successfully.


In [14]:
verified_df = pd.read_csv("Level_3_Verified_Train_Schedule.csv")

print("Verified dataset shape:", verified_df.shape)

Verified dataset shape: (186074, 15)


# Level 3 — Key Findings & Observations

## Data Quality Findings

1. **Missing Values**
   - No null/NaN values were found in the dataset.
   - Therefore, no rows were removed and no null-value imputation was required.
   - `00:00:00` schedule values were not treated as null values because they represent boundary schedule entries in certain train routes.

2. **Duplicate Records**
   - Complete duplicate records were checked across all columns.
   - The dataset contains **0 duplicate records**.
   - No records were removed during duplicate handling.
   - Repeated train numbers were retained because each train contains multiple station records representing its route.

3. **Station Order Validation**
   - Station sequence was validated using the `SN` column.
   - All **11,113 trains** have station sequence numbers in increasing order.
   - No incorrectly ordered train routes were identified.
   - Therefore, no station records required reordering.

4. **Final Dataset Validation**
   - Total records: **186,074**
   - Total attributes: **15**
   - Unique trains: **11,113**
   - Missing values: **0**
   - Duplicate records: **0**

5. **Verified Dataset**
   - After completing the quality checks, the verified dataset was saved as:
     `Level_3_Verified_Train_Schedule.csv`
   - This verified dataset will be used as the input for the next level of analysis and visualization.